In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
PROJECT_DIR = Path(
    r"F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs"
FORECAST_DIR = OUTPUT_DIR / "forecasts"
FIGURE_DIR = OUTPUT_DIR / "figures"

# Create folders if they do not exist
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FORECAST_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROCESSED_DIR / "train_features.csv"
TEST_PATH = PROCESSED_DIR / "test_features.csv"

print("Project directory:", PROJECT_DIR)
print("Model directory:", MODEL_DIR)
print("Training file:", TRAIN_PATH)
print("Testing file:", TEST_PATH)

Project directory: F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand
Model directory: F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand\models
Training file: F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand\data\processed\train_features.csv
Testing file: F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand\data\processed\test_features.csv


In [3]:
if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"Training file not found: {TRAIN_PATH}\n"
        "Run 04_feature_engineering.ipynb first."
    )

if not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Testing file not found: {TEST_PATH}\n"
        "Run 04_feature_engineering.ipynb first."
    )

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Training and testing datasets loaded!")
print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

display(train_df.head())

Training and testing datasets loaded!
Training shape: (245, 28)
Testing shape: (62, 28)


,DATE,BOOKING_COUNT,year,month,day,day_of_week,day_name,week_of_year,quarter,is_weekend,is_month_start,is_month_end,lag_1,lag_2,lag_3,lag_7,lag_14,lag_28,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_std_7,demand_change_1,demand_change_7,day_sin,day_cos,month_sin,month_cos
0,2026-02-05,0.0,2026,2,5,3,Thursday,6,1,0,0,0,0.0,0.0,0.0,0.0,0.0,13691.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.433884,-0.900969,0.5,0.866025
1,2026-02-06,0.0,2026,2,6,4,Friday,6,1,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,-0.433884,-0.900969,0.5,0.866025
2,2026-02-07,0.0,2026,2,7,5,Saturday,6,1,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,-0.974928,-0.222521,0.5,0.866025
3,2026-02-08,1913.0,2026,2,8,6,Sunday,6,1,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,-0.781831,0.623490,0.5,0.866025
4,2026-02-09,0.0,2026,2,9,0,Monday,7,1,0,0,0,1913.0,0.0,0.0,0.0,0.0,0.0,637.666667,273.285714,136.642857,723.046037,1913.0,1913.0,0.000000,1.000000,0.5,0.866025


In [4]:
import pandas as pd

# Display available columns
print("Available columns:", train_df.columns.tolist())

# Clean column names
train_df.columns = (
    train_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Date column candidates
date_candidates = [
    "bkdate", "date", "booking_date",
    "bookingdate", "ds", "book_date"
]

# Demand column candidates
demand_candidates = [
    "booking_count", "daily_booking_demand",
    "demand", "bookings", "y",
    "total_bookings", "booking_demand",
    "no_of_bookings", "number_of_bookings",
    "total_booking", "count"
]

# Automatically detect columns
date_col = next(
    (col for col in date_candidates
     if col in train_df.columns),
    None
)

demand_col = next(
    (col for col in demand_candidates
     if col in train_df.columns),
    None
)

# Detect possible demand columns if not found
if demand_col is None:
    possible_cols = [
        col for col in train_df.columns
        if any(word in col for word in
               ["booking", "demand", "count", "total"])
        and col != date_col
    ]

    if len(possible_cols) == 1:
        demand_col = possible_cols[0]

# Validate columns
if date_col is None:
    raise ValueError(
        f"Date column not found. Available columns: "
        f"{train_df.columns.tolist()}"
    )

if demand_col is None:
    raise ValueError(
        f"Demand column not identified. "
        f"Possible columns: {possible_cols}"
    )

# Convert date and demand columns
train_df[date_col] = pd.to_datetime(
    train_df[date_col], errors="coerce"
)

train_df[demand_col] = pd.to_numeric(
    train_df[demand_col], errors="coerce"
)

# Remove rows with missing date or demand
train_df = train_df.dropna(
    subset=[date_col, demand_col]
).copy()

# Display results
print("\nDate column:", date_col)
print("Demand column:", demand_col)
print("Training data shape:", train_df.shape)

display(train_df.head())

Available columns: ['DATE', 'BOOKING_COUNT', 'year', 'month', 'day', 'day_of_week', 'day_name', 'week_of_year', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_3', 'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7', 'demand_change_1', 'demand_change_7', 'day_sin', 'day_cos', 'month_sin', 'month_cos']

Date column: date
Demand column: booking_count
Training data shape: (245, 28)


,date,booking_count,year,month,day,day_of_week,day_name,week_of_year,quarter,is_weekend,is_month_start,is_month_end,lag_1,lag_2,lag_3,lag_7,lag_14,lag_28,rolling_mean_3,rolling_mean_7,rolling_mean_14,rolling_std_7,demand_change_1,demand_change_7,day_sin,day_cos,month_sin,month_cos
0,2026-02-05,0.0,2026,2,5,3,Thursday,6,1,0,0,0,0.0,0.0,0.0,0.0,0.0,13691.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.433884,-0.900969,0.5,0.866025
1,2026-02-06,0.0,2026,2,6,4,Friday,6,1,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,-0.433884,-0.900969,0.5,0.866025
2,2026-02-07,0.0,2026,2,7,5,Saturday,6,1,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,-0.974928,-0.222521,0.5,0.866025
3,2026-02-08,1913.0,2026,2,8,6,Sunday,6,1,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,-0.781831,0.623490,0.5,0.866025
4,2026-02-09,0.0,2026,2,9,0,Monday,7,1,0,0,0,1913.0,0.0,0.0,0.0,0.0,0.0,637.666667,273.285714,136.642857,723.046037,1913.0,1913.0,0.000000,1.000000,0.5,0.866025


In [5]:
import pandas as pd

# Check available columns
print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

# Automatically find the date column
date_candidates = [
    "BKDATE", "date", "Date", "DATE",
    "booking_date", "ds"
]

# Match column names without case sensitivity
train_cols = {col.strip().lower(): col for col in train_df.columns}
test_cols = {col.strip().lower(): col for col in test_df.columns}

date_col = next(
    (train_cols[name.lower()] for name in date_candidates
     if name.lower() in train_cols),
    None
)

if date_col is None:
    raise ValueError("Date column not found in train_df.")

if date_col.strip().lower() not in test_cols:
    raise ValueError("Date column not found in test_df.")

test_date_col = test_cols[date_col.strip().lower()]

# Convert dates
train_df[date_col] = pd.to_datetime(
    train_df[date_col], errors="coerce"
)

test_df[test_date_col] = pd.to_datetime(
    test_df[test_date_col], errors="coerce"
)

# Remove invalid dates
train_df = train_df.dropna(subset=[date_col]).copy()
test_df = test_df.dropna(subset=[test_date_col]).copy()

# Sort by date
train_df = train_df.sort_values(
    date_col
).reset_index(drop=True)

test_df = test_df.sort_values(
    test_date_col
).reset_index(drop=True)

# Display results
print("Dates converted and sorted successfully.")
print("Date column:", date_col)

print(
    "Training dates:",
    train_df[date_col].min(),
    "to",
    train_df[date_col].max()
)

print(
    "Testing dates:",
    test_df[test_date_col].min(),
    "to",
    test_df[test_date_col].max()
)

Train columns: ['date', 'booking_count', 'year', 'month', 'day', 'day_of_week', 'day_name', 'week_of_year', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_3', 'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7', 'demand_change_1', 'demand_change_7', 'day_sin', 'day_cos', 'month_sin', 'month_cos']
Test columns: ['DATE', 'BOOKING_COUNT', 'year', 'month', 'day', 'day_of_week', 'day_name', 'week_of_year', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_3', 'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7', 'demand_change_1', 'demand_change_7', 'day_sin', 'day_cos', 'month_sin', 'month_cos']
Dates converted and sorted successfully.
Date column: date
Training dates: 2026-02-05 00:00:00 to 2026-10-07 00:00:00
Testing dates: 2026-10-08 00:00:00 to 2026-12-08 00:00:00


In [6]:
feature_columns = [
    "year",
    "month",
    "day",
    "day_of_week",
    "week_of_year",
    "quarter",
    "is_weekend",
    "is_month_start",
    "is_month_end",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_3",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_std_7",
    "demand_change_1",
    "demand_change_7",
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos"
]

missing_train = [
    col for col in feature_columns
    if col not in train_df.columns
]

missing_test = [
    col for col in feature_columns
    if col not in test_df.columns
]

if missing_train or missing_test:
    raise ValueError(
        f"Missing training features: {missing_train}\n"
        f"Missing testing features: {missing_test}\n"
        "Check Notebook 04."
    )

print("All required features are available.")
print("Number of features:", len(feature_columns))

All required features are available.
Number of features: 25


In [7]:
print("TRAIN DATA")
print(train_df.head())
print(train_df.columns.tolist())

print("\nTEST DATA")
print(test_df.head())
print(test_df.columns.tolist())

TRAIN DATA
        date  booking_count  year  month  day  day_of_week  day_name  week_of_year  quarter  is_weekend  is_month_start  is_month_end   lag_1  lag_2  lag_3  lag_7  lag_14   lag_28  rolling_mean_3  rolling_mean_7  rolling_mean_14  rolling_std_7  demand_change_1  demand_change_7   day_sin   day_cos  month_sin  month_cos
0 2026-02-05            0.0  2026      2    5            3  Thursday             6        1           0               0             0     0.0    0.0    0.0    0.0     0.0  13691.0        0.000000        0.000000         0.000000       0.000000              0.0              0.0  0.433884 -0.900969        0.5   0.866025
1 2026-02-06            0.0  2026      2    6            4    Friday             6        1           0               0             0     0.0    0.0    0.0    0.0     0.0      0.0        0.000000        0.000000         0.000000       0.000000              0.0              0.0 -0.433884 -0.900969        0.5   0.866025
2 2026-02-07            0.0  

In [ ]:
import pandas as pd
from pathlib import Path

# Load original Excel dataset
DATA_PATH = Path("../data/raw/BK_AUG2026.xlsx")

df = pd.read_excel(DATA_PATH)

# Display dataset information
print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

In [ ]:

print("=" * 55)
print("LOGISTICS BOOKING FORECASTING - MODEL TRAINING")
print("=" * 55)

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))

# Training period
print("\nTraining period:")
if date_col in train_df.columns:
    print(train_df[date_col].min())
    print("to")
    print(train_df[date_col].max())
else:
    print("Training dates unavailable")

# Testing period
print("\nTesting period:")
if date_col in test_df.columns:
    print(test_df[date_col].min())
    print("to")
    print(test_df[date_col].max())
else:
    print("Test dates unavailable in test_df")

# Display trained models safely
print("\nModels trained:")

if "trained_models" in globals():
    for name in trained_models:
        print("-", name)
elif "models" in globals():
    for name in models:
        print("-", name)
else:
    print("Model dictionary not found")

# Display model performance
print("\nModel performance:")

if "comparison" in globals():
    display(comparison.round(3))
else:
    print("Comparison results not found")

# Display best model
print("\nBest model:")

if "best_model_name" in globals():
    print(best_model_name)
else:
    print("Best model name not found")

# Display saved model files
print("\nSaved model files:")

if "MODEL_DIR" in globals():
    from pathlib import Path

    model_dir = Path(MODEL_DIR)

    if model_dir.exists():
        model_files = list(model_dir.glob("*.joblib"))

        if model_files:
            for file in model_files:
                print(file)
        else:
            print("No saved .joblib models found")
    else:
        print("Model directory does not exist")
else:
    print("MODEL_DIR is not defined")

# Output paths
print("\nPredictions:")
if "PREDICTION_PATH" in globals():
    print(PREDICTION_PATH)
else:
    print("Prediction path not defined")

print("\nMetrics:")
if "METRICS_PATH" in globals():
    print(METRICS_PATH)
else:
    print("Metrics path not defined")

print("\nReport completed!")

In [ ]:

import pandas as pd

# Automatically find the target column (case-insensitive)
demand_col = next(
    (col for col in test_df.columns
     if col.strip().lower() == "booking_count"),
    None
)

if demand_col is None:
    raise ValueError("BOOKING_COUNT column not found.")

# Create training and testing targets
train_target_col = next(
    (col for col in train_df.columns
     if col.strip().lower() == "booking_count"),
    None
)

if train_target_col is None:
    raise ValueError("BOOKING_COUNT missing in train_df.")

y_train = pd.to_numeric(
    train_df[train_target_col],
    errors="coerce"
)

y_test = pd.to_numeric(
    test_df[demand_col],
    errors="coerce"
)

# Check missing values
print("Missing values in training features:")
display(X_train.isnull().sum())

print("\nMissing values in testing features:")
display(X_test.isnull().sum())

print("\nMissing training targets:", y_train.isnull().sum())
print("Missing testing targets:", y_test.isnull().sum())

print("\nTraining target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

In [ ]:

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

# Define models
models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

print("Models created successfully!")

for name in models:
    print("-", name)

In [ ]:
predictions = {}

for name, model in trained_models.items():
    predictions[name] = model.predict(X_test)

print("Predictions generated successfully!")

for name, values in predictions.items():
    print(f"{name}: {len(values)} predictions")

In [ ]:
def calculate_metrics(actual, predicted):
    mae = mean_absolute_error(actual, predicted)

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    r2 = (
        r2_score(actual, predicted)
        if len(actual) >= 2
        else np.nan
    )

    actual_array = np.asarray(actual)
    predicted_array = np.asarray(predicted)

    nonzero = actual_array != 0

    if nonzero.any():
        mape = np.mean(
            np.abs(
                (actual_array[nonzero] - predicted_array[nonzero])
                / actual_array[nonzero]
            )
        ) * 100
    else:
        mape = np.nan

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE (%)": mape
    }

results = {}

for name, predicted in predictions.items():
    results[name] = calculate_metrics(
        y_test,
        predicted
    )

comparison = pd.DataFrame(results).T
comparison.index.name = "Model"

print("Model evaluation completed!")
display(comparison.round(3))

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

print("Models defined successfully!")

In [ ]:
import matplotlib.pyplot as plt

# Find the date column, regardless of capitalization
date_col = next(
    col for col in test_df.columns
    if col.strip().lower() == "date"
)

# Get test dates in the same order as X_test
test_dates = pd.to_datetime(
    test_df.loc[X_test.index, date_col]
)

plt.figure(figsize=(15, 7))

plt.plot(
    test_dates,
    y_test,
    marker="o",
    linewidth=2,
    label="Actual Demand"
)

for name, predicted in predictions.items():
    plt.plot(
        test_dates,
        predicted,
        marker=".",
        label=name
    )

plt.title("Actual vs Predicted Booking Demand")
plt.xlabel("Date")
plt.ylabel("Booking Demand")
plt.legend()
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "model_training_predictions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
prediction_results = pd.DataFrame({
    date_col: test_dates.to_numpy(),
    "actual": y_test.to_numpy()
})

for name, predicted in predictions.items():
    safe_name = (
        name.lower()
        .replace(" ", "_")
    )

    prediction_results[f"{safe_name}_prediction"] = predicted

display(prediction_results.head(10))

In [ ]:
PREDICTION_PATH = (
    FORECAST_DIR / "model_training_predictions.csv"
)

prediction_results.to_csv(
    PREDICTION_PATH,
    index=False
)

print("Predictions saved successfully!")
print("File:", PREDICTION_PATH)

In [ ]:
METRICS_PATH = (
    FORECAST_DIR / "model_training_metrics.csv"
)

comparison.to_csv(METRICS_PATH)

print("Model metrics saved successfully!")
print("File:", METRICS_PATH)

display(comparison.round(3))

In [ ]:
for name, model in trained_models.items():
    filename = (
        name.lower().replace(" ", "_") + ".joblib"
    )

    model_path = MODEL_DIR / filename

    joblib.dump(model, model_path)

    print(f"Saved {name} to {model_path}")

In [ ]:
FEATURES_PATH = MODEL_DIR / "feature_columns.json"

import json

with open(FEATURES_PATH, "w", encoding="utf-8") as file:
    json.dump(feature_columns, file, indent=4)

print("Feature names saved successfully!")
print("File:", FEATURES_PATH)

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

print("Models defined:", list(models.keys()))

In [ ]:
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model

print("Training completed!")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = []

predictions = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred)
    })

comparison = pd.DataFrame(results).set_index("Model")

display(comparison)

best_model_name = comparison["MAE"].idxmin()
best_model = trained_models[best_model_name]

print("Selected model:", best_model_name)
print("MAE:", round(comparison.loc[best_model_name, "MAE"], 3))

In [ ]:
BASELINE_PATH = (
    FORECAST_DIR / "baseline_model_metrics.csv"
)

if BASELINE_PATH.exists():
    baseline_comparison = pd.read_csv(
        BASELINE_PATH,
        index_col=0
    )

    combined_comparison = pd.concat(
        [
            baseline_comparison[["MAE", "RMSE"]],
            comparison[["MAE", "RMSE"]]
        ],
        axis=0
    )

    print("Baseline and machine learning model comparison:")
    display(combined_comparison.round(3))

    combined_comparison.to_csv(
        FORECAST_DIR / "all_model_comparison.csv"
    )

else:
    print(
        "Baseline metrics file not found. "
        "Run 05_baseline_model.ipynb to create it."
    )

In [ ]:
print("=" * 55)
print("LOGISTICS BOOKING FORECASTING - MODEL TRAINING")
print("=" * 55)

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))

print("\nModels trained:")
for name in trained_models:
    print("-", name)

print("\nModel performance:")
display(comparison.round(3))

print("\nModel with lowest test MAE:", best_model_name)

print("\nSaved model files:")
for name in trained_models:
    filename = name.lower().replace(" ", "_") + ".joblib"
    print(MODEL_DIR / filename)

print("\nPredictions:", PREDICTION_PATH)
print("Metrics:", METRICS_PATH)

print("\nModel training completed successfully!")